In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

data = fetch_kddcup99(percent10=True, as_frame=True)
df = data.frame

df.columns = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot",
              "num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations",
              "num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count",
              "serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
              "srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
              "dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
              "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
              "dst_host_srv_rerror_rate","label"]

df['label'] = df['label'].apply(lambda x: 'normal' if x == 'normal.' else 'attack')

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

df = df.groupby('label').apply(lambda x: x.sample(2500, random_state=42)).reset_index(drop=True)

for col in ['protocol_type','service','flag']:
    df[col] = LabelEncoder().fit_transform(df[col])

df['byte_ratio'] = df['src_bytes'] / (df['dst_bytes'] + 1)
df['total_count'] = df['count'] + df['srv_count']
df['error_rate'] = (df['serror_rate'] + df['rerror_rate']) / 2

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

models = ['Decision Tree', 'KNN', 'Logistic Regression']

accuracy = [0.995, 0.994, 0.994]
precision = [0.980, 0.975, 0.985]
recall = [0.995, 0.995, 0.990]

x = np.arange(len(models))
width = 0.25

plt.figure(figsize=(9,5.5))

plt.bar(x - width, accuracy, width, label='Accuracy')
plt.bar(x, precision, width, label='Precision')
plt.bar(x + width, recall, width, label='Recall')

plt.title('Comparison of Accuracy, Precision, and Recall')
plt.xlabel('Models')
plt.ylabel('Score')

plt.xticks(x, models)
plt.ylim(0.85, 1.01)

plt.grid(axis='y', linestyle='-', alpha=0.6)
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_30188\3135132746.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label').apply(lambda x: x.sample(2500, random_state=42)).reset_index(drop=True)


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)